In [0]:
from pyspark.sql.functions import * 
from pyspark.sql.types import * 
from delta.tables import DeltaTable

In [0]:
cust_address = spark.read.format("parquet")\
                .load("abfss://bronze@intechaccstorage.dfs.core.windows.net/CustomerAddress")
cust_address.limit(5).display()

CustomerID,AddressID,AddressType,rowguid,ModifiedDate,_rescued_data
29485,1086,Main Office,16765338-dbe4-4421-b5e9-3836b9278e63,2007-09-01 00:00:00.0000000,null
29486,621,Main Office,22b3e910-14af-4ed5-8b4d-23bbe757414d,2005-09-01 00:00:00.0000000,null
29489,1069,Main Office,a095c88b-d7e6-4178-a078-2eca44214801,2005-07-01 00:00:00.0000000,null
29490,887,Main Office,f12e1702-d897-4035-b614-0fe2c72168a9,2006-09-01 00:00:00.0000000,null
29492,618,Main Office,5b3b3eb2-3f43-47ed-a20c-23697dabf23b,2006-12-01 00:00:00.0000000,null


In [0]:
cust_address = cust_address.withColumnRenamed("CustomerID", "Cust_id")\
                           .withColumnRenamed("AddressID", "Address_id")\
                           .withColumnRenamed("AddressType", "Address_type")\
                           .withColumnRenamed("ModifiedDate", "Modified_date")

cust_address = cust_address.withColumn("Modified_date", to_date(col("Modified_date")))

cust_address = cust_address.drop("rowguid", "_rescued_data")
                               

In [0]:
cust_address.columns

['Cust_id', 'Address_id', 'Address_type', 'Modified_date']

In [0]:
%sql
CREATE TABLE IF NOT EXISTS databricksintech.silver.customer_address
(
    Cust_id STRING,
    Address_id STRING,
    Address_type STRING,
    Modified_date DATE
) 
USING DELTA
LOCATION 'abfss://silver@intechaccstorage.dfs.core.windows.net/customer_address';

In [0]:
# Reference target Delta table
silver_table = DeltaTable.forName(spark, "databricksintech.silver.customer_address")

# Execute the Merge (Upsert)
silver_table.alias("target").merge(
    source = cust_address.alias("source"),
    condition = (
        "target.Cust_id = source.Cust_id AND "
        "target.Address_id = source.Address_id"
        
    )
).whenMatchedUpdateAll() \
 .whenNotMatchedInsertAll() \
 .execute()

print("Done!")

Done!
